# Querying the PSDI lakehouse with Trino

This notebook is a hands-on tutorial for querying the PSDI lakehouse through Trino.

It covers:

- connecting to Trino with PSDI authentication,
- discovering schemas and tables,
- running simple SQL queries,
- and working through examples using the parsed [OMol25](adf) dataset.

## 1. Install and import dependencies

We will need the Trino Python client to fetch data and `pandas` to manage datasets.

In [ ]:
!pip install -q trino pandas

Now let's import the required components:

In [ ]:
from pprint import pprint  # Used for clearer output in the tutorial

import pandas as pd

from trino.dbapi import connect
from trino.auth import OAuth2Authentication

## 1. Connect to Trino

Run the cell below to open a connection. At the first run, you will be prompted to authenticate via the PSDI Authentification system in your browser.

In [ ]:
TRINO_HOST = "trino-staging.psdi.ac.uk"

try:
    conn = connect(
        host=TRINO_HOST,
        port=443,
        http_scheme="https",
        auth=OAuth2Authentication(),
        catalog="psdi",
        request_timeout=300,
    )
    
    cursor = conn.cursor()

    # Validate the connection
    cursor.execute("SHOW SCHEMAS FROM psdi")
    cursor.fetchone()
    
    print("\nConnected to Trino successfully!")

except Exception as e:
    print("\nFailed to connect to Trino")
    raise

## 2. Discover what is available in the lakehouse

### 2.1 Terminology

In the lakehouse, data is organised in the following hierarchy:

**catalog** --> **schema** --> **table**

- **catalog**: a data source for Trino (in our case, `psdi`)
- **schema**: corresponds to a *dataset* (e.g. `omol25`, `materials_project`)
- **table**: a structured collection of records within a dataset (e.g. `alloy_pairs`)

---

**Catalog**

The catalog name in our case is `psdi`. It is specified when establishing the connection (`catalog="psdi"`), so it can usually be omitted in queries.

---

**Schema**

The proper way to refer to a schema is `<catalog_name>.<schema_name>`  
(or just `<schema_name>` if the catalog is already set).

Examples of schema names:
- `psdi.omol25`
- `psdi.materials_project`

---

**Table**

The proper way to refer to a table is `<catalog_name>.<schema_name>.<table_name>`  
(or `<schema_name>.<table_name>` if the catalog is already set).

Examples of table names:
- `psdi.omol25.omol25`
- `psdi.materials_project.absorption_flattened`
- `psdi.materials_project.alloys_flattened`

### 2.2 General pattern for executing commands

To execute SQL queries, you first need to create a **cursor object**, which represents an active SQL session. The recommended approach is to use a context manager:

```python
with conn.cursor() as cursor:
    cursor.execute("<SQL coomand>")
    result = cursor.fetchall()
```
    
Within the context manager:
- `cursor.execute("<SQL command>")`: sends a SQL command to Trino for execution.
- `cursor.fetchall()`: retrieves all results returned by Trino.
---

### 2.3 Listing available schemas

The following code lists all available schemas (i.e. datasets), excluding system ones:

In [ ]:
with conn.cursor() as cursor:
    cursor.execute("SHOW SCHEMAS FROM psdi")
    schemas = cursor.fetchall()

    # Exclude system schemas
    exclude = {"information_schema", "system"}
    
    datasets = [s[0] for s in schemas if s[0] not in exclude]
    pprint(datasets)

Note that we haven't inluded the catalog name, `psdi`, in the query as it was specified when establishing the connection. However, you can optionally include it by using the command `SHOW SCHEMAS FROM psdi`. 

---

### 2.4 Listing available tables within a schema

Now let's have a closer look at the tables inside a particular schema, for example `materials_project`.

In [ ]:
with conn.cursor() as cursor:
    cursor.execute("SHOW TABLES FROM psdi.materials_project")
    
    tables = cursor.fetchall()
    pprint(tables)

Note that some schemas contain only one table, for example `omol25`:

In [ ]:
with conn.cursor() as cursor:
    cursor.execute("SHOW TABLES FROM psdi.omol25")
    
    tables = cursor.fetchall()
    pprint(tables)

---
### 2.5 Explore columns from a specific table

To inspect column names and types in the specific table, you can run the `DESCRIBE` command followed by `<schema_name>.<table_name>`, for example:

In [ ]:
with conn.cursor() as cursor:
    cursor.execute("DESCRIBE psdi.omol25.omol25")
    
    columns = cursor.fetchall()
    pprint(columns)

`DESCRIBE` always returns 4 fields:
- name (column name)
- type (data type)
- extra (additional metadata, if available)
- comment (description of the column, if provided)

The last two fields are often empty if no additional metadata or comments are defined.

---
### 2.6 Previewing data from a table

To inspect a table, it is often useful to display a small number of rows.  
This can be done using the `SELECT` statement together with `LIMIT`.

For example, the following query returns a single row from the `omol25.omol25` table:

In [ ]:
with conn.cursor() as cursor:
    cursor.execute("SELECT * FROM psdi.omol25.omol25 LIMIT 1")
    
    rows = cursor.fetchall()
    pprint(rows)

---
### 2.7 Counting rows in a table

To count rows, you can use the following command 
```python
SELECT COUNT(*) FROM <catalog>.<schema>.<table>
```

For example:

In [ ]:
with conn.cursor() as cursor:
    cursor.execute("SELECT COUNT(*) FROM psdi.omol25.omol25")
    rows = cursor.fetchone()  # fetchone() returns a single tuple like (1234,)
    
    print("Row count:", rows[0])

## 3. Basic SQL patterns

Here is a good general recipe for exploring any lakehouse table:

1. List all available schemas using `SHOW SCHEMAS FROM <catalog>`.
2. List all available tables in a given schema using `SHOW TABLES FROM <catalog>.<schema>`.
3. Inspect the columns of a table using `DESCRIBE <catalog>.<schema>.<table>`.
4. Count rows in the table using `SELECT COUNT(*) FROM <catalog>.<schema>.<table>`.
5. Preview data from the table using `SELECT * FROM <catalog>.<schema>.<table> LIMIT 1`.
6. Then write analytical queries of your interest. The most commonly used SQL commands include `SELECT`, `COUNT(*)`, `GROUP BY`, and `ORDER BY`.

---

### Understanding common SQL patterns

**`SELECT`**  
Used to retrieve data from a table. You can select all columns or only specific ones. In the following examples, we retrieve 5 rows.

```sql
SELECT *
FROM psdi.materials_project.alloy_pairs_flattened
LIMIT 5
```
```sql
SELECT column_name_a, column_name_b
FROM psdi.materials_project.alloy_pairs_flattened
LIMIT 5
```

**`COUNT(*)`**
Counts the number of rows in a table or within a group.
```sql
SELECT COUNT(*)
FROM psdi.materials_project.alloy_pairs_flattened
```

**`GROUP BY`**
Groups rows that share the same values in specified columns.
Often used together with aggregation functions like `COUNT`, `AVG`, etc.

Example: count how many entries exist for each element:
```sql
SELECT column_name_a, COUNT(*)
FROM psdi.materials_project.alloy_pairs_flattened
GROUP BY column_name_a
```

**`ORDER BY`**
Sorts the results of a query.
```sql
SELECT column_name_a, COUNT(*) AS count
FROM psdi.materials_project.alloy_pairs_flattened
GROUP BY column_name_a
ORDER BY count DESC
```

- ASC --> ascending order (default)
- DESC --> descending order

**Putting it all together**
These commands are often combined to answer questions about the data.

For example, to find the most common elements:
```sql
SELECT column_name_a, COUNT(*) AS count
FROM psdi.materials_project.alloy_pairs_flattened
GROUP BY column_name_a
ORDER BY count DESC
LIMIT 10
```

This query:
1. selects a column (`SELECT`)
2. counts occurrences (`COUNT(*)`)
3. groups results (`GROUP BY`)
4. sorts them (`ORDER BY`)
5. limits the output (`LIMIT`)

## 4. OMol25 example

Below are some example analytical queries for the OMol25 dataset.

### 4.1 Distribution of number of atoms

This query computes the distribution of molecules by their number of atoms.
It groups rows by the `num_atoms` column and counts how many molecules fall into each group.

- `SELECT`: Specifies the columns to return. Here, it also calculates the number of occurrences using `COUNT(*)`.
- `FROM`: Indicates the source table (`psdi.omol25.omol25`).
- `GROUP BY num_atoms`: Aggregates rows that have the same number of atoms.
- `COUNT(*)`: Counts how many rows (molecules) are in each group.
- `ORDER BY num_atoms`: Sorts the results in ascending order of the number of atoms.

In [ ]:
with conn.cursor() as cursor:
    cursor.execute("""
    SELECT num_atoms, COUNT(*)
    FROM psdi.omol25.omol25
    GROUP BY num_atoms
    ORDER BY num_atoms
    """)
    
    for row in cursor.fetchall():
        print(row)

---
### 4.2 Charge distribution

This query calculates the distribution of molecules by their formal charge.
It groups molecules by the charge column and counts how many molecules fall into each charge category.

- `SELECT`: Specifies the columns to return. Here, it selects the `charge` column and also calculates the number of occurrences using `COUNT(*)`.
- `FROM`: Indicates the source table (`psdi.omol25.omol25`).
- `GROUP BY charge`: Collects all rows with the same charge value.
- `COUNT(*)`: Computes how many molecules are in each charge group.
- `ORDER BY charge`: Sorts results from lowest to highest charge.

In [ ]:
with conn.cursor() as cursor:
    cursor.execute("""
    SELECT charge, COUNT(*)
    FROM psdi.omol25.omol25
    GROUP BY charge
    ORDER BY charge
    """)
    
    for row in cursor.fetchall():
        print(row)

---
### 4.3 Most common compositions

The query retrieves the 20 most frequent values of the composition column from the `psdi.omol25.omol25` table.

It uses several standard SQL commands:

- `SELECT`: Specifies the columns to return. Here, it selects `composition` and calculates the number of occurrences using `COUNT(*)`, aliased as `count`.
- `FROM`: Indicates the source table (`psdi.omol25.omol25`).
- `GROUP BY`: Groups rows by composition so that the count can be computed for each unique value.
- `ORDER BY`: Sorts the results in descending order based on the computed count. `DESC` means the results are ordered from highest to lowest.
- `LIMIT 5`: Restricts the output to the top 5 results.

In [ ]:
with conn.cursor() as cursor:
    cursor.execute("""
    SELECT composition, COUNT(*) AS count
    FROM psdi.omol25.omol25
    GROUP BY composition
    ORDER BY count DESC
    LIMIT 5
    """)
    
    for row in cursor.fetchall():
        print(row)

---
### 4.4 NL energy stats

This query calculates basic statistics for the `nl_energy` column in the `psdi.omol25.omol25` table. Specifically, it returns the minimum, maximum, and average values, excluding any missing (`NULL`) entries.

1. `SELECT`: Specifies the values to return. Here, it computes aggregate statistics and returns them as aliases `(min_nl, max_nl, avg_nl)`:
    - `MIN(nl_energy)` --> the smallest value
    - `MAX(nl_energy)` --> the largest value
    - `AVG(nl_energy)` --> the average value
2. `FROM`: Indicates the source table (`psdi.omol25.omol25`).
3. `WHERE`: Filters out rows where `nl_energy` is `NULL`, ensuring that only valid numeric values are used in the calculations.

In [ ]:
with conn.cursor() as cursor:
    cursor.execute("""
    SELECT
        MIN(nl_energy) AS min_nl,
        MAX(nl_energy) AS max_nl,
        AVG(nl_energy) AS avg_nl
    FROM psdi.omol25.omol25
    WHERE nl_energy IS NOT NULL
    """)
    
    print('Min, max and average energies:')
    print(cursor.fetchone())

---
### 4.5 Using pandas for post-processing of Trino query results

When working with large datasets, it is inefficient to load all rows into memory. Instead, you should perform heavy computations (such as grouping and aggregation) in Trino, and use `pandas` only for lightweight post-processing of the reduced result set.

The query below retrieves a small sample of rows (non-null `nl_energy` values) from the `psdi.omol25.omol25` table. The result is then loaded into a pandas DataFrame for further analysis.

In [ ]:
with conn.cursor() as cursor:
    cursor.execute("""
    SELECT composition, nl_energy
    FROM psdi.omol25.omol25
    WHERE nl_energy IS NOT NULL
    LIMIT 100
    """)

    # Load results into a pandas DataFrame
    df = pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])

# Print top rows
print(df.head())

# Compute average nl_energy per composition (on the sampled data)
print(df.groupby('composition')['nl_energy'].mean())

## 5. Close connection to Trino
It is good practice to close your connection to Trino when it is not not longer needed, as each open connection holds server-side resources. You can do this by running:


In [ ]:
conn.close()

An alternative approach is using context manager, which will automatically close the connection. To do that, you can use the following auxiliary function:

In [ ]:
def get_connection(host: str):
    try:
        conn = connect(
            host=TRINO_HOST,
            port=443,
            http_scheme="https",
            auth=OAuth2Authentication(),
            catalog="psdi",
            request_timeout=300,
        )

        # Validate the connection
        with conn.cursor() as cursor:
            cursor.execute("SHOW SCHEMAS")
            cursor.fetchone()
            
        print("\nConnected to Trino successfully!")
        return conn
    except Exception as e:
        print("\nFailed to connect to Trino")
        raise

Then you can use the following pattern for quering the lakehouse:

In [ ]:
TRINO_HOST = "trino-dev.psdi.ac.uk"

with get_connection(TRINO_HOST) as conn:
    with conn.cursor() as cursor:
        cursor.execute("SHOW SCHEMAS FROM psdi")
        schemas = cursor.fetchall()
        print(schemas)

## 6. Troubleshooting

If something fails:

- confirm the Keycloak login completed successfully,
- run `SHOW SCHEMAS FROM <catalog>` to list all available shemas(datasets),
- run `SHOW TABLES FROM <catalog>.<schema>` to check the actual table name,
- and then inspect columns by running `DESCRIBE <catalog>.<schema_name>.<table_name>`.